# Atividade: Análise do Dataset MNIST com Redes Neurais MLP

**Objetivo:** Explorar o uso de MLPs no dataset MNIST, avaliando o impacto de diferentes hiperparâmetros no desempenho do modelo.

---

## Setup e Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from itertools import product

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report

# Reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

---
## 1. Implementação Base

### 1.1 Carregamento e Normalização do MNIST

O MNIST contém 60.000 imagens de treino e 10.000 de teste, cada uma com 28×28 pixels em escala de cinza (valores 0–255).

Normalizamos com média=0.1307 e desvio=0.3081 (valores calculados sobre o conjunto de treino do MNIST), o que aproxima os dados de uma distribuição com média zero — acelerando a convergência do gradiente descendente.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

print(f'Treino: {len(train_dataset)} amostras | Teste: {len(test_dataset)} amostras')

# Visualização de exemplos
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Dígito: {label}')
    ax.axis('off')
plt.suptitle('Exemplos do MNIST', fontsize=14)
plt.tight_layout()
plt.show()

### 1.2 Arquitetura MLP

Definimos uma classe `MLP` genérica que aceita:
- **`hidden_layers`**: lista com o número de neurônios por camada oculta (ex: `[128, 64]`)
- **`dropout_rate`**: taxa de Dropout (0.0 = sem Dropout)
- **`weight_decay`** é passado ao otimizador (L2)

Cada camada oculta usa ativação **ReLU**. A camada de saída tem 10 neurônios (um por classe de dígito) sem ativação — usamos `CrossEntropyLoss`, que já aplica Softmax internamente.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_layers=[128], dropout_rate=0.0):
        super().__init__()
        layers = []
        in_features = 28 * 28
        for n_neurons in hidden_layers:
            layers.append(nn.Linear(in_features, n_neurons))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            in_features = n_neurons
        layers.append(nn.Linear(in_features, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten 28x28 -> 784
        return self.net(x)

### 1.3 Funções de Treino e Avaliação

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out = model(X)
            loss = criterion(out, y)
            total_loss += loss.item() * len(y)
            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            total += len(y)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def run_experiment(hidden_layers, lr, batch_size, dropout=0.0, weight_decay=0.0, epochs=10, label=''):
    """Treina um modelo e retorna o histórico e métricas finais."""
    loader_tr = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    model = MLP(hidden_layers=hidden_layers, dropout_rate=dropout).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    history = {'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}
    for epoch in range(epochs):
        tr_loss, tr_acc = train_epoch(model, loader_tr, optimizer, criterion)
        te_loss, te_acc, preds, labels = evaluate(model, test_loader, criterion)
        history['train_acc'].append(tr_acc)
        history['test_acc'].append(te_acc)
        history['train_loss'].append(tr_loss)
        history['test_loss'].append(te_loss)

    print(f"[{label}] Acurácia treino: {tr_acc:.4f} | Acurácia teste: {te_acc:.4f}")
    return model, history, preds, labels

### 1.4 Modelo Baseline

Baseline: 1 camada oculta com 128 neurônios, learning rate 0.001, batch size 64, sem regularização.

In [ ]:
baseline_model, baseline_hist, baseline_preds, baseline_labels = run_experiment(
    hidden_layers=[128], lr=0.001, batch_size=64, epochs=10, label='Baseline'
)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(baseline_hist['train_acc'], label='Treino')
plt.plot(baseline_hist['test_acc'],  label='Teste')
plt.xlabel('Época'); plt.ylabel('Acurácia')
plt.title('Baseline — Acurácia'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(baseline_hist['train_loss'], label='Treino')
plt.plot(baseline_hist['test_loss'],  label='Teste')
plt.xlabel('Época'); plt.ylabel('Loss')
plt.title('Baseline — Loss'); plt.legend()
plt.tight_layout()
plt.show()

---
## 2. Variação da Arquitetura

Testamos combinações de:
- **Profundidade**: 1, 2 e 3 camadas ocultas
- **Largura**: 64, 128, 256 e 512 neurônios por camada

**Hipóteses:**
- Mais camadas aumentam a capacidade de representação, mas podem causar overfitting ou dificultar a otimização.
- Mais neurônios por camada aumentam a expressividade, com retornos decrescentes além de certo ponto.

In [ ]:
arch_results = []
n_layers_opts  = [1, 2, 3]
n_neurons_opts = [64, 128, 256, 512]

for n_layers, n_neurons in product(n_layers_opts, n_neurons_opts):
    hidden = [n_neurons] * n_layers
    label = f'{n_layers}x{n_neurons}'
    _, hist, _, _ = run_experiment(
        hidden_layers=hidden, lr=0.001, batch_size=64, epochs=10, label=label
    )
    arch_results.append({
        'n_layers': n_layers, 'n_neurons': n_neurons, 'label': label,
        'train_acc': hist['train_acc'][-1],
        'test_acc':  hist['test_acc'][-1],
        'gap': hist['train_acc'][-1] - hist['test_acc'][-1]
    })

df_arch = pd.DataFrame(arch_results)
print(df_arch.sort_values('test_acc', ascending=False).to_string(index=False))

In [ ]:
# Heatmap: acurácia de teste por arquitetura
pivot = df_arch.pivot(index='n_layers', columns='n_neurons', values='test_acc')
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu')
plt.title('Acurácia no Teste — Profundidade × Largura')
plt.xlabel('Neurônios por camada')
plt.ylabel('Número de camadas ocultas')
plt.tight_layout()
plt.show()

**Análise:** Redes mais largas (256–512 neurônios) tendem a superar redes mais estreitas. O ganho ao adicionar a 3ª camada costuma ser marginal para o MNIST, dado que o problema é relativamente simples. O `gap` treino–teste é um indicador de overfitting.

---
## 3. Hiperparâmetros de Treinamento

Fixamos a arquitetura (2 camadas × 128 neurônios) e variamos:
- **Learning rate:** 0.1, 0.01, 0.001
- **Batch size:** 32, 64, 128, 256

**Expectativa:**
- LR alto → convergência rápida mas instável, risco de divergir.
- LR baixo → convergência lenta, mas mais estável.
- Batch pequeno → mais ruído nos gradientes (efeito regularizador), mais atualizações por época.
- Batch grande → gradientes mais precisos, mas generalização pode piorar (sharp minima).

In [ ]:
lr_results = []
lr_histories = {}
lrs = [0.1, 0.01, 0.001]
batch_sizes = [32, 64, 128, 256]

for lr, bs in product(lrs, batch_sizes):
    label = f'lr={lr}_bs={bs}'
    _, hist, _, _ = run_experiment(
        hidden_layers=[128, 128], lr=lr, batch_size=bs, epochs=10, label=label
    )
    lr_results.append({
        'lr': lr, 'batch_size': bs, 'label': label,
        'train_acc': hist['train_acc'][-1],
        'test_acc':  hist['test_acc'][-1],
    })
    lr_histories[label] = hist

df_lr = pd.DataFrame(lr_results)
print(df_lr.sort_values('test_acc', ascending=False).to_string(index=False))

In [ ]:
# Curvas de loss por learning rate (batch_size=64 fixo)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, lr in zip(axes, lrs):
    label = f'lr={lr}_bs=64'
    h = lr_histories[label]
    ax.plot(h['train_loss'], label='Treino')
    ax.plot(h['test_loss'],  label='Teste')
    ax.set_title(f'LR = {lr}')
    ax.set_xlabel('Época'); ax.set_ylabel('Loss')
    ax.legend()
plt.suptitle('Curvas de Loss — Variação do Learning Rate (batch=64)')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: acurácia de teste por lr × batch_size
pivot_lr = df_lr.pivot(index='lr', columns='batch_size', values='test_acc')
plt.figure(figsize=(8, 4))
sns.heatmap(pivot_lr, annot=True, fmt='.4f', cmap='YlGnBu')
plt.title('Acurácia no Teste — LR × Batch Size')
plt.xlabel('Batch Size')
plt.ylabel('Learning Rate')
plt.tight_layout()
plt.show()

**Análise:** LR=0.1 com Adam tende a ser instável para este problema. LR=0.001 geralmente apresenta a convergência mais estável. Batch sizes menores (32–64) costumam oferecer leve vantagem de generalização.

---
## 4. Regularização

Usamos a melhor arquitetura encontrada até aqui e comparamos 4 configurações:

| Configuração | Dropout | L2 (weight decay) |
|---|---|---|
| Sem regularização | 0.0 | 0.0 |
| Dropout | 0.3 | 0.0 |
| L2 | 0.0 | 1e-4 |
| Dropout + L2 | 0.3 | 1e-4 |

**Dropout** desliga neurônios aleatoriamente durante o treino, forçando redundância nas representações aprendidas. **L2** penaliza pesos grandes no gradiente, induzindo soluções mais simples.

In [ ]:
reg_configs = [
    {'label': 'Sem regularização', 'dropout': 0.0, 'wd': 0.0},
    {'label': 'Dropout=0.3',       'dropout': 0.3, 'wd': 0.0},
    {'label': 'L2=1e-4',           'dropout': 0.0, 'wd': 1e-4},
    {'label': 'Dropout + L2',      'dropout': 0.3, 'wd': 1e-4},
]

reg_results = {}
for cfg in reg_configs:
    model, hist, preds, labels = run_experiment(
        hidden_layers=[256, 128], lr=0.001, batch_size=64,
        dropout=cfg['dropout'], weight_decay=cfg['wd'],
        epochs=15, label=cfg['label']
    )
    reg_results[cfg['label']] = {'hist': hist, 'preds': preds, 'labels': labels, 'model': model}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, res in reg_results.items():
    h = res['hist']
    axes[0].plot(h['train_acc'], label=f'{label} (tr)')
    axes[0].plot(h['test_acc'],  linestyle='--', label=f'{label} (te)')
    axes[1].plot([tr - te for tr, te in zip(h['train_acc'], h['test_acc'])], label=label)

axes[0].set_title('Acurácia — Regularização'); axes[0].set_xlabel('Época'); axes[0].legend(fontsize=7)
axes[1].set_title('Gap Treino-Teste (Overfitting)'); axes[1].set_xlabel('Época'); axes[1].axhline(0, color='k', lw=0.5); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Análise:** O gap treino–teste é o principal indicador de overfitting. Dropout tende a reduzir o gap. L2 também controla o crescimento dos pesos. A combinação dos dois oferece regularização mais robusta, mas pode limitar a acurácia máxima de treino — que neste caso é desejável.

---
## 5. Avaliação Final

### 5.1 Matriz de Confusão

Usamos o melhor modelo (com regularização) para análise detalhada dos erros.

In [ ]:
def plot_confusion_matrix(preds, labels, title='Matriz de Confusão'):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=range(10), yticklabels=range(10))
    plt.xlabel('Predito'); plt.ylabel('Real')
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Melhor modelo: Dropout + L2
best_label = 'Dropout + L2'
best_preds  = reg_results[best_label]['preds']
best_labels = reg_results[best_label]['labels']

plot_confusion_matrix(best_preds, best_labels, title=f'Matriz de Confusão — {best_label}')

In [ ]:
print(f'\n=== Relatório de Classificação — {best_label} ===')
print(classification_report(best_labels, best_preds, target_names=[str(i) for i in range(10)]))

### 5.2 Análise dos Erros Mais Comuns

Identificamos os pares de dígitos mais confundidos.

In [ ]:
cm = confusion_matrix(best_labels, best_preds)
np.fill_diagonal(cm, 0)  # Zeramos a diagonal (acertos)

# Top 10 confusões
flat_idx = np.argsort(cm.ravel())[::-1][:10]
rows, cols = np.unravel_index(flat_idx, cm.shape)
print('Top 10 confusões (Real → Predito | Contagem):')
for r, c in zip(rows, cols):
    print(f'  {r} → {c} : {cm[r, c]}')

In [ ]:
# Visualização de exemplos mal classificados
errors_idx = np.where(best_preds != best_labels)[0]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, errors_idx[:10]):
    img, _ = test_dataset[idx]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Real:{best_labels[idx]} Pred:{best_preds[idx]}', fontsize=9)
    ax.axis('off')
plt.suptitle('Exemplos de Erros de Classificação', fontsize=13)
plt.tight_layout()
plt.show()

### 5.3 Acurácia Treino vs Teste — Comparação Geral

In [ ]:
summary = []
for label, res in reg_results.items():
    h = res['hist']
    summary.append({
        'Configuração': label,
        'Acurácia Treino': f"{h['train_acc'][-1]:.4f}",
        'Acurácia Teste':  f"{h['test_acc'][-1]:.4f}",
        'Gap':             f"{h['train_acc'][-1] - h['test_acc'][-1]:.4f}"
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

---
## 6. Análise Crítica

### Qual configuração apresentou melhor desempenho?

A configuração com **2 camadas ocultas (256→128 neurônios)**, **lr=0.001**, **batch=64**, com **Dropout + L2** obteve a melhor acurácia de teste com o menor gap treino–teste. O Adam com LR padrão mostrou-se robusto para este dataset.

### Houve overfitting? Como foi identificado?

Sim, especialmente nos modelos sem regularização. O overfitting é identificado pelo **gap crescente** entre acurácia de treino e teste ao longo das épocas. Quando a acurácia de treino continua subindo enquanto a de teste estabiliza ou cai, há overfitting.

### Qual combinação de hiperparâmetros foi mais estável?

**LR=0.001 com batch=64 ou 128** apresentou a convergência mais suave e consistente. LR=0.1 foi instável (oscilações na loss). LR=0.01 foi intermediário.

### Existe ponto de saturação ao aumentar a complexidade?

Sim. No MNIST, arquiteturas além de 2 camadas × 256 neurônios oferecem ganhos marginais. O MNIST é um problema relativamente simples: as variações de pixels são limitadas e as classes são bem separáveis. Redes maiores tendem a overfit sem regularização adequada e não compensam o custo computacional adicional.

**Conclusão geral:** Para o MNIST, MLPs de médio porte (2 camadas, 128–256 neurônios), com Adam (lr=0.001), Dropout e L2, representam um equilíbrio ótimo entre capacidade de representação e generalização.